In [45]:
from os import error
import yaml
import numpy as np
import pandas as pd
import csv
import time
import json
import logging
import argparse

errors = []

In [46]:
parser = argparse.ArgumentParser(description="MLOps Batch Job")

parser.add_argument("--input", required=True, help="Input CSV file")
parser.add_argument("--config", required=True, help="Config YAML file")
parser.add_argument("--output", required=True, help="Output JSON file")
parser.add_argument("--log-file", required=True, help="Log file")

args = parser.parse_args()

print("Input file:", args.input)
print("Config file:", args.config)
print("Output file:", args.output)
print("Log file:", args.log_file)


usage: colab_kernel_launcher.py [-h] --input INPUT --config CONFIG --output
                                OUTPUT --log-file LOG_FILE
colab_kernel_launcher.py: error: the following arguments are required: --input, --config, --output, --log-file
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2230, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
argparse.ArgumentError: the following arguments are required: --input, --config, --output, --log-file

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_271/3020941297.py", line 8, in <cell line: 0>
    args = parser.parse_args()
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 1904, in parse_args
    args, argv = self.par

TypeError: object of type 'NoneType' has no len()

In [ ]:
logging.basicConfig(
    filename='run.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='a'
)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)   # 🔥 IMPORTANT

# Record the start time
start_time = time.perf_counter()
logger.info("Job start timestamp: %f", start_time)

In [47]:
def validate_config(config_file):
  with open(config_file, 'r') as f:
    config = yaml.safe_load(f)

  if 'seed' not in config or 42 != config['seed']:
    raise valueError("Misssing Value of seed")
  if 'window'not in  config or 5 != config['window']:
    raise valueError("Missing Value of window")
  if 'version' not in config or 'v1' not in config['version']:
    raise valueError("Missing Value of version")

  logger.info("Config loaded and validated")
  return config

try:
  config = validate_config('config.yaml')
  print("Config file loaded successfully")
  seed = config['seed']
  np.random.seed(seed)
  print(f"Seed set to {seed}")
  print("Random Numbers Generated from given seed")
  print(np.random.rand(4))
except ValueError as e:
  print(e)
  errors.append(e)
  logging.error(f"Error occurred: {str(e)}")
except FileNotFoundError as e:
  print(e)
  errors.append(e)
  logging.error(f"Error occurred: {str(e)}")

INFO:__main__:Config loaded and validated


Config file loaded successfully
Seed set to 42
Random Numbers Generated from given seed
[0.37454012 0.95071431 0.73199394 0.59865848]


In [48]:
def rolling_mean_cal(df):
  window = config['window']
  df['close'] = pd.to_numeric(df['close'], errors='coerce')
  df['rolling_mean'] = df['close'].rolling(window=window).mean()
  df = df.dropna()
  logger.info(f"Rolling Mean calculated")
  return df


In [49]:
def signal_genration(df):
  df = df.reset_index(drop=True)
  df['signal'] = np.where(df['close'] > df['rolling_mean'], 1, 0)
  logger.info(f"Signal is generated")
  return df

In [50]:
def latency_ms_cal(start_time, end_time):
  latency = end_time - start_time
  latency_ms = round(latency * 1000)
  return latency_ms

In [51]:
filepath = 'data.csv'
expected_column= ['close']

try:
  csv_file = open(filepath, 'r')
  dialect = csv.Sniffer().sniff(csv_file.read(1024))
  csv_file.seek(0)

  df = pd.read_csv(filepath, header=None)
  df = df[0].str.split(',', expand=True)
  df.columns = ['timestamp','open','high','low','close','volume_btc','volume_usd']
  actual_columns = df.columns.tolist()
  print(f"Actual Column: {actual_columns}")

  if 'close' not in df.columns:
    raise ValueError("Missing 'close' column in CSV")

  df = rolling_mean_cal(df)
  print("Rolling Mean Calculated Successfully")
  df = signal_genration(df)
  print("Signal Generated Successfully")

  rows_processed= df.shape[0]
  print(f"Rows Processed: {rows_processed}")
  logger.info(f"Rows Processed: {rows_processed}")

  signal_rate = df['signal'].mean()
  signal_rate = round(signal_rate, 4 )
  print(f"Signal Rate: {signal_rate}")

  # Record the end time
  end_time = time.perf_counter()
  latency_ms = latency_ms_cal(start_time, end_time)
  print(f"Latency: {latency_ms} seconds")



except csv.Error as e:
  print(e)
  errors.append(e)
  logging.error(f"Error occurred: {str(e)}")
except ValueError as e:
  print(e)
  errors.append(e)
  logging.error(f"Error occurred: {str(e)}")
except Exception as e:
  print(e)
  errors.append(e)
  logging.error(f"Error occurred: {str(e)}")
except FileNotFoundError as e:
  print(e)
  errors.append(e)
  logging.error(f"Error occurred: {str(e)}")


INFO:__main__:Rolling Mean calculated
INFO:__main__:Signal is generated
INFO:__main__:Rows Processed: 9996


Actual Column: ['timestamp', 'open', 'high', 'low', 'close', 'volume_btc', 'volume_usd']
Rolling Mean Calculated Successfully
Signal Generated Successfully
Rows Processed: 9996
Signal Rate: 0.4991
Latency: 463 seconds


In [52]:
data = {
      "Success output":
    {
      "version":config['version'] ,
      "rows_processed": rows_processed,
      "metric": "signal_rate",
      "value": signal_rate,
      "latency_ms": latency_ms,
      "seed":config['seed'],
      "status": "success"
    },
     "Error output": {
      "version": config['version'],
      "status": "error",
      "error_message": errors
    }

}

try:
    with open("metrics.json", "w") as f:
        json.dump(data, f)
    logger.info(f"Metrics summary generated successfully")
except Exception as e:
    print(e)
    logging.error(f"Error occurred: {str(e)}")




INFO:__main__:Metrics summary generated successfully


In [53]:
if len(errors) == 0:
  logger.info(f"Job completed successfully Without Error")
else:
  logger.error(f"Error occured while completing job")


INFO:__main__:Job completed successfully Without Error
